# Lab 02 — Feedforward networks and backpropagation

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist), binary 0-vs-1 classification. We will explicitly compare a shallow linear baseline with a nonlinear MLP and verify one gradient numerically.


> **Experiment contract:** do not change the intervention before writing your prediction. Every run reports the same metric and budget so the comparison is interpretable.


## The math behind the experiment

A small network first computes a forward pass:

$$
z^{(1)}=W^{(1)}x+b^{(1)},\qquad h=\sigma(z^{(1)}),\qquad \hat{y}=W^{(2)}h+b^{(2)}.
$$

The loss tells us how wrong the prediction is. Backpropagation applies the chain rule to determine how much each parameter contributed to that loss:

$$
\frac{\partial L}{\partial w}=\frac{\partial L}{\partial \hat{y}}\frac{\partial \hat{y}}{\partial h}\frac{\partial h}{\partial z}\frac{\partial z}{\partial w}.
$$

Gradient descent then updates a parameter by

$$
w\leftarrow w-\eta\frac{\partial L}{\partial w}.
$$

The finite-difference check in this lab approximates the same derivative numerically, giving us a powerful debugging test for automatic differentiation.

In [ ]:
!pip -q install datasets scikit-learn torch pandas matplotlib
from datasets import load_dataset
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score
seed=11; torch.manual_seed(seed)
tr=load_dataset("ylecun/mnist",split="train[:4000]"); te=load_dataset("ylecun/mnist",split="test[:1000]")
def xy(ds):
 X=np.array([np.asarray(x).reshape(-1) for x in ds["image"]],np.float32)/255.; y=np.array(ds["label"]); m=(y<2); return torch.tensor(X[m]),torch.tensor(y[m],dtype=torch.float32)
Xtr,ytr=xy(tr); Xte,yte=xy(te); print(Xtr.shape,Xte.shape)

## Step 1 — Prediction

Predict that a nonlinear hidden layer will improve over a linear classifier because it can transform the input into a representation that makes the decision easier. Predict that shuffled labels will destroy held-out accuracy.

In [ ]:
import torch.nn as nn
class Linear(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Linear(784,1)
 def forward(self,x): return self.net(x).squeeze(-1)
class MLP(nn.Module):
 def __init__(self,width=64): super().__init__(); self.net=nn.Sequential(nn.Linear(784,width),nn.ReLU(),nn.Linear(width,1))
 def forward(self,x): return self.net(x).squeeze(-1)
def train(model,epochs=8,lr=1e-2):
 opt=torch.optim.SGD(model.parameters(),lr=lr); lossfn=nn.BCEWithLogitsLoss()
 for _ in range(epochs):
  opt.zero_grad(); loss=lossfn(model(Xtr),ytr); loss.backward(); opt.step()
 with torch.no_grad():
  a=((torch.sigmoid(model(Xtr))>.5)==ytr.bool()).float().mean().item(); b=((torch.sigmoid(model(Xte))>.5)==yte.bool()).float().mean().item()
 return a,b
rows=[]
for name,m in [("linear",Linear()),("MLP-64",MLP(64)),("MLP-128",MLP(128))]:
 a,b=train(m); rows.append([name,a,b])
pd.DataFrame(rows,columns=["model","train_accuracy","test_accuracy"])

## Step 2 — Numerical gradient check

Autograd should agree with a finite-difference approximation. This is a debugging experiment: if the two disagree materially, do not trust later training results.

In [ ]:
torch.manual_seed(3)
w=torch.tensor([0.7],dtype=torch.double,requires_grad=True); x=torch.tensor([2.0],dtype=torch.double); y=torch.tensor([1.0],dtype=torch.double)
loss=((w*x-y)**2).mean(); loss.backward(); analytic=w.grad.item()
eps=1e-6
with torch.no_grad():
 lp=(( (w+eps)*x-y)**2).mean().item(); lm=(( (w-eps)*x-y)**2).mean().item()
numeric=(lp-lm)/(2*eps)
print({"analytic_gradient":analytic,"numeric_gradient":numeric,"absolute_error":abs(analytic-numeric)})

## Step 3 — Width intervention and shuffled-label control

Keep optimizer and epochs fixed. Change only hidden width. Then permute training labels and repeat the same MLP. Compare train vs test behavior.

In [ ]:
rng=np.random.default_rng(seed); ysh=torch.tensor(rng.permutation(ytr.numpy()),dtype=torch.float32)
def train_labels(width,labels):
 m=MLP(width); opt=torch.optim.SGD(m.parameters(),lr=1e-2); lf=nn.BCEWithLogitsLoss()
 for _ in range(8): opt.zero_grad(); lf(m(Xtr),labels).backward(); opt.step()
 with torch.no_grad(): return ((torch.sigmoid(m(Xtr))>.5)==labels.bool()).float().mean().item(),((torch.sigmoid(m(Xte))>.5)==yte.bool()).float().mean().item()
rows=[]
for w in [16,64,128]:
 a,b=train_labels(w,ytr); rows.append(["real",w,a,b]); a,b=train_labels(w,ysh); rows.append(["shuffled",w,a,b])
pd.DataFrame(rows,columns=["labels","width","train_accuracy","test_accuracy"])

## Read the graphs — capacity and generalization

The left plot compares training and test accuracy for the real-label and shuffled-label runs. High training accuracy alone is not evidence of learning: a model may memorize arbitrary labels. The right plot isolates the numerical gradient check, comparing the derivative from backpropagation with a finite-difference estimate.

For the gradient check, the two bars should be almost indistinguishable. The printed absolute error is the more precise diagnostic.

In [ ]:
import matplotlib.pyplot as plt

width_results = pd.DataFrame(rows, columns=["labels", "width", "train_accuracy", "test_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label_name, group in width_results.groupby("labels"):
    axes[0].plot(group["width"], group["test_accuracy"], marker="o", label=f"{label_name}: test")
    axes[0].plot(group["width"], group["train_accuracy"], marker="x", linestyle="--", label=f"{label_name}: train")
axes[0].set_title("Capacity and generalization")
axes[0].set_xlabel("Hidden-layer width")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)
axes[0].grid(alpha=.25)

axes[1].bar(["Backprop", "Finite difference"], [analytic, numeric])
axes[1].set_title("Gradient check")
axes[1].set_ylabel("Estimated derivative")
axes[1].ticklabel_format(axis="y", style="plain")
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## Conclusion / answer key

Expected: the nonlinear MLP should generally outperform the linear model; numerical and analytic gradients should agree closely; shuffled labels can produce high training accuracy with poor test accuracy when capacity is sufficient.

**Interpretation rule:** do not say “deeper is better” unless the parameter budget and training budget make that comparison fair.

### Research extension
Compare depth and width at approximately equal parameter count, then report mean ± standard deviation over three seeds.